In [ ]:
"""
fitters_nb.ipynb

Tests the fitter object, meant to 
fit all models to a given session.

Author: Stellina X. Ao
Created: 2026-03-05
Last Modified: 2026-05-04
Python Version: 3.11
"""

from squiggs.neuron_viewer import NeuronViewer
from squiggs.renderers import FitRenderer
from sg.fitter import LVMFamily
from sg.eval_models import plot_summary

import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt
from pathlib import Path

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

# Fit

In [ ]:
subj_id = "MM012"  # MR82
sess_id = "20231219_130847"  # "20251027_152036"

In [ ]:
family = LVMFamily(
    subj_id=subj_id,
    sess_id=sess_id,
    n_latents_mult=1,
    n_latents_addt=1,
    sanity_check=0,
    task_vars=[
        "response",
        "rewarded",
        "block_side",
        "response_prev",
        "rewarded_prev",
    ],
    n_splines=5,
    tpre=0.5,
    tpost=1,
)
family.fit_all()
family.eval()

In [ ]:
from sg.eval_models import plot_r2_comp

plot_r2_comp(family.res_taskvar, family.res_affine, family.qi)

In [ ]:
from squiggs.renderers import PETHRasterRenderer
from core.data import get_psths_cond, get_choice_ts
from utils.paths import FIGURES_DIR

reg = "DLS"
mode = "response"

renderer = PETHRasterRenderer(
    event_times=get_choice_ts(family.trial_data, mode=mode),
    spike_times=family.spike_times[reg],
    peths=get_psths_cond(family.psths[reg], family.trial_data, mode=mode),
    pres=0.5,
    posts=1,
    binwidth_s=25 / 1000,
    s=0.2,
    linewidths=0.2,
    save_subdir=Path("peths") / subj_id / sess_id / reg / mode,
)

nv = NeuronViewer(
    num_units=family.psths[reg].shape[0], render_func=renderer, fig_dir=FIGURES_DIR
)

In [ ]:
plot_summary(family, family.mod_gain, family.response, label="choice", mode="gain")

In [ ]:
renderer = FitRenderer(
    family.mod_affine,
    x=family.test_dl.dataset[:],
    y=family.robs,
    dfs=family.test_dl.dataset[:]["dfs"][:, family.cids].detach().cpu().numpy(),
    save_subdir=Path("model_fits") / subj_id / sess_id / "taskvar",
)

nv = NeuronViewer(num_units=renderer.y.shape[1], render_func=renderer)